In [1]:
import pandas as pd
import numpy as np

# LOAD THE PROCESSED DATASET FROM NOTEBOOK 01 (already merged + has_identity added)
df = pd.read_csv('/kaggle/input/datasets/hamaz911/notebook-2-dataset/train_merged.csv')
print(df.shape)

(590540, 435)


In [2]:
# 1. TRANSACTION AMOUNT - BASIC STATS BY FRAUD STATUS
print(df.groupby('isFraud')['TransactionAmt'].describe())

            count        mean         std    min     25%   50%    75%  \
isFraud                                                                 
0        569877.0  134.511665  239.395078  0.251  43.970  68.5  120.0   
1         20663.0  149.244779  232.212163  0.292  35.044  75.0  161.0   

               max  
isFraud             
0        31937.391  
1         5191.000  


In [3]:
# 2. CHECK HOW EXTREME THE OUTLIERS ACTUALLY ARE
for pct in [0.95, 0.99, 0.999]:
    print(f"{int(pct*100)}th percentile:")
    print(df.groupby('isFraud')['TransactionAmt'].quantile(pct))
    print()

95th percentile:
isFraud
0    435.0
1    500.0
Name: TransactionAmt, dtype: float64

99th percentile:
isFraud
0    1104.0
1     994.0
Name: TransactionAmt, dtype: float64

99th percentile:
isFraud
0    2775.38068
1    2680.33462
Name: TransactionAmt, dtype: float64



In [4]:
# 3. INSPECT TransactionDT RANGE
print(df['TransactionDT'].describe())
print("\nRange in days:", (df['TransactionDT'].max() - df['TransactionDT'].min()) / (60*60*24))


count    5.905400e+05
mean     7.372311e+06
std      4.617224e+06
min      8.640000e+04
25%      3.027058e+06
50%      7.306528e+06
75%      1.124662e+07
max      1.581113e+07
Name: TransactionDT, dtype: float64

Range in days: 181.99920138888888


In [5]:
# 4. DERIVE TIME-BASED FEATURES FROM TransactionDT
df['hour_of_day'] = (df['TransactionDT'] / 3600) % 24
df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7

print(df[['TransactionDT', 'hour_of_day', 'day_of_week']].head())

   TransactionDT  hour_of_day  day_of_week
0          86400     0.000000     1.000000
1          86401     0.000278     1.000012
2          86469     0.019167     1.000799
3          86499     0.027500     1.001146
4          86506     0.029444     1.001227


In [6]:
# 5. FRAUD RATE BY HOUR OF DAY
df['hour_bucket'] = df['hour_of_day'].astype(int)
fraud_by_hour = df.groupby('hour_bucket')['isFraud'].agg(['mean', 'count'])
fraud_by_hour['mean'] = fraud_by_hour['mean'] * 100
print(fraud_by_hour)

                  mean  count
hour_bucket                  
0             3.137981  37795
1             3.131384  32797
2             3.748317  26732
3             3.831362  20802
4             5.189029  14839
5             7.030203   9701
6             7.774263   6007
7            10.610151   3704
8             9.301428   2591
9             8.995563   2479
10            5.321202   3627
11            3.881646   6827
12            3.043932  12451
13            2.288949  20315
14            2.421632  28328
15            2.539945  33859
16            2.951057  38698
17            3.153009  40723
18            3.523139  41639
19            3.473822  42115
20            3.427313  41782
21            3.400495  41641
22            3.269404  41139
23            3.699717  39949


In [7]:
# 6. DOUBLE-CHECK: DOES THE HOUR PATTERN HOLD IN BOTH HALVES OF THE TIME PERIOD?
midpoint = df['TransactionDT'].median()
df['time_half'] = np.where(df['TransactionDT'] < midpoint, 'first_half', 'second_half')

for half in ['first_half', 'second_half']:
    print(f"\n--- {half} ---")
    subset = df[df['time_half'] == half]
    result = subset.groupby('hour_bucket')['isFraud'].agg(['mean', 'count'])
    result['mean'] = result['mean'] * 100
    print(result.loc[[6, 7, 8, 9, 13, 14]])  # just the peak-fraud and low-fraud hours


--- first_half ---
                 mean  count
hour_bucket                 
6            6.599132   3455
7            7.593123   2094
8            8.518006   1444
9            9.501845   1084
13           2.008796   8413
14           2.231174  12908

--- second_half ---
                  mean  count
hour_bucket                  
6             9.365204   2552
7            14.534161   1610
8            10.287707   1147
9             8.602151   1395
13            2.486977  11902
14            2.581064  15420


In [8]:
# 7. FRAUD RATE BY ProductCD
fraud_by_product = df.groupby('ProductCD')['isFraud'].agg(['mean', 'count'])
fraud_by_product['mean'] = fraud_by_product['mean'] * 100
print(fraud_by_product.sort_values('mean', ascending=False))

# 8. FRAUD RATE BY card4 (visa/mastercard/etc.)
fraud_by_card4 = df.groupby('card4')['isFraud'].agg(['mean', 'count'])
fraud_by_card4['mean'] = fraud_by_card4['mean'] * 100
print("\n", fraud_by_card4.sort_values('mean', ascending=False))

# 9. FRAUD RATE BY card6 (debit/credit)
fraud_by_card6 = df.groupby('card6')['isFraud'].agg(['mean', 'count'])
fraud_by_card6['mean'] = fraud_by_card6['mean'] * 100
print("\n", fraud_by_card6.sort_values('mean', ascending=False))

                mean   count
ProductCD                   
C          11.687269   68519
S           5.899553   11628
H           4.766231   33024
R           3.782594   37699
W           2.039939  439670

                       mean   count
card4                             
discover          7.728161    6651
visa              3.475610  384767
mastercard        3.433095  189217
american express  2.869837    8328

                      mean   count
card6                            
credit           6.678480  148986
debit            2.426251  439938
charge card      0.000000      15
debit or credit  0.000000      30


In [9]:
# 10. FIND V-COLUMNS AND CHECK THEIR CORRELATION WITH isFraud
v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
print(f"Number of V columns: {len(v_cols)}")

v_correlations = df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').sort_values(ascending=False)
print("\nTop 15 positively correlated V-columns:")
print(v_correlations.head(15))

print("\nTop 15 negatively correlated V-columns:")
print(v_correlations.tail(15))

Number of V columns: 339

Top 15 positively correlated V-columns:
V257    0.383060
V246    0.366878
V244    0.364129
V242    0.360590
V201    0.328005
V200    0.318783
V189    0.308219
V188    0.303582
V258    0.297151
V45     0.281832
V158    0.278066
V156    0.275952
V149    0.273282
V228    0.268861
V44     0.260376
Name: isFraud, dtype: float64

Top 15 negatively correlated V-columns:
V160   -0.062338
V159   -0.062660
V150   -0.063965
V36    -0.069875
V11    -0.071950
V10    -0.074658
V35    -0.077116
V70    -0.100900
V30    -0.103088
V69    -0.104521
V91    -0.105453
V29    -0.107852
V90    -0.110168
V49    -0.110920
V48    -0.115786
Name: isFraud, dtype: float64


In [10]:
# 11. CHECK IF TOP V-COLUMNS ARE REDUNDANT (CORRELATED WITH EACH OTHER)
top_v_cols = ['V257', 'V246', 'V244', 'V242', 'V201', 'V200', 'V189', 'V188', 'V258', 'V45']
print(df[top_v_cols].corr().round(2))

      V257  V246  V244  V242  V201  V200  V189  V188  V258   V45
V257  1.00  0.91  0.76  0.75  0.73  0.75  0.52  0.52  0.78  0.74
V246  0.91  1.00  0.83  0.83  0.67  0.68  0.57  0.57  0.74  0.67
V244  0.76  0.83  1.00  0.97  0.54  0.53  0.66  0.66  0.71  0.45
V242  0.75  0.83  0.97  1.00  0.53  0.53  0.65  0.63  0.71  0.46
V201  0.73  0.67  0.54  0.53  1.00  0.94  0.80  0.74  0.64  0.67
V200  0.75  0.68  0.53  0.53  0.94  1.00  0.71  0.78  0.63  0.66
V189  0.52  0.57  0.66  0.65  0.80  0.71  1.00  0.90  0.46  0.38
V188  0.52  0.57  0.66  0.63  0.74  0.78  0.90  1.00  0.44  0.37
V258  0.78  0.74  0.71  0.71  0.64  0.63  0.46  0.44  1.00  0.67
V45   0.74  0.67  0.45  0.46  0.67  0.66  0.38  0.37  0.67  1.00


In [11]:
# 12. DO TOP V-COLUMNS CORRELATE WITH TransactionAmt ITSELF?
print(df[top_v_cols + ['TransactionAmt']].corr()['TransactionAmt'])

# 13. GROUP COLUMNS BY IDENTICAL MISSINGNESS PATTERN
missing_fingerprint = df.isnull().apply(lambda row: tuple(row), axis=0)  # this can be slow on 435 cols, let's do it smarter:
missing_matrix = df.isnull()
fingerprint_groups = missing_matrix.T.groupby(missing_matrix.T.apply(tuple, axis=1)).size()
print(f"Number of distinct missingness patterns across all columns: {len(fingerprint_groups)}")
print(fingerprint_groups.sort_values(ascending=False).head(10))

V257             -0.009878
V246             -0.017490
V244             -0.013688
V242             -0.004766
V201             -0.000919
V200             -0.005478
V189              0.003040
V188             -0.007555
V258              0.004002
V45              -0.015402
TransactionAmt    1.000000
Name: TransactionAmt, dtype: float64
Number of distinct missingness patterns across all columns: 70
(True, True, True, True, False, True, True, True, False, True, False, False, True, True, True, True, False, False, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, False, True, True, False, False, True, False, True, False, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, False, True, True, True, T

In [12]:
# SAVE NOTEBOOK 02 OUTPUT (WITH GENUINE TIME FEATURES, MINUS EDA SCAFFOLDING)
df_export = df.drop(columns=['time_half'])
df_export.to_parquet('/kaggle/working/train_with_time_features.parquet', index=False)
print("Saved:", df_export.shape)

Saved: (590540, 438)
